# Lab 12: CSP Backtracking for Smart Film Production Casting

## Overview

In this lab, you will implement a Constraint Satisfaction Problem (CSP) solver using:

- Basic Backtracking
- Backtracking with Inference
- Minimum Remaining Values (MRV)
- Degree Heuristic
- Least Constraining Value (LCV)

The system will assign actors to movie scenes while satisfying production constraints.

You will also:
- Read input from files
- Use Object-Oriented Programming (OOP)
- Produce formatted console and file outputs
- Implement AI search heuristics

---

# Real World Scenario

A streaming company is shooting a futuristic science fiction series.

Each scene requires a specific skill such as:
- action
- comedy
- stunt
- sci-fi

Actors possess different skills.

The AI system must assign actors to scenes while ensuring:
- each scene gets a compatible actor
- one actor is not assigned to multiple conflicting scenes
- constraints are satisfied efficiently

This simulates:
- AI scheduling systems
- production planning software
- resource allocation problems

---

# Input File Format

You will use two input files.

## actors.txt

Each line contains:

ActorName: skill1,skill2,skill3

Example:

Ava: action,english

---

## scenes.txt

Each line contains:

SceneName: required_skill

Example:

Scene1: action

---


# Expected Learning Outcomes

By the end of this lab students will be able to:

- Implement recursive backtracking
- Apply CSP heuristics
- Use Forward Checking inference
- Design OOP-based AI systems
- Read and write files in Python
- Solve real-world scheduling problems

## Step 01: Make Imports

In [1]:
from collections import defaultdict

## Step 02: Run Input Handler (Do NOT Modify)

In [2]:
class InputHandler:

    @staticmethod
    def load_data(actor_file, scene_file):

        actors = {}
        scenes = {}

        with open(actor_file, "r") as f:

            for line in f.readlines():

                parts = line.strip().split(":")

                actor = parts[0]

                skills = parts[1].split(",")

                actors[actor] = skills

        with open(scene_file, "r") as f:

            for line in f.readlines():

                parts = line.strip().split(":")

                scene = parts[0]

                requirement = parts[1].strip()

                scenes[scene] = requirement

        return actors, scenes

## Step 03: Create CSP Class
In this step you will:

- store variables
- store domains
- store constraints
- implement consistency checking

In [3]:
class FilmCSP:

    def __init__(self):

        self.variables = []

        self.domains = {}

        self.constraints = defaultdict(list)

    def add_variable(self, variable, domain):

        self.variables.append(variable)
        self.domains[variable] = list(domain)

        for other_scene in self.variables:

            if other_scene != variable:

                for a in domain:

                    if a in self.domains[other_scene]:

                        self.add_constraint(variable, other_scene)
                        break

    def add_constraint(self, var1, var2):

        if var2 not in self.constraints[var1]:
            self.constraints[var1].append(var2)

        if var1 not in self.constraints[var2]:
            self.constraints[var2].append(var1)

    def is_consistent(self, variable, value, assignment):

        for other in self.constraints[variable]:

            if other in assignment:

                if assignment[other] == value:
                    return False

        return True

## Step 04: Implement MRV Heuristic

In this step you will:

- select the variable with the smallest remaining domain
- reduce unnecessary branching

In [4]:
class MRVHeuristic:

    @staticmethod
    def select_unassigned_variable(csp, assignment):

        selected = None
        minimum = None

        for scene in csp.variables:

            if scene not in assignment:

                size = len(csp.domains[scene])

                if selected is None or size < minimum:
                    selected = scene
                    minimum = size

        return selected

## Step 05: Implement Degree Heuristic

In this step you will:

- choose the variable connected to most neighbors
- prioritize difficult variables first

In [5]:
class DegreeHeuristic:

    @staticmethod
    def select_variable(csp, assignment):

        result = None
        highest = -1

        for scene in csp.variables:

            if scene not in assignment:

                connections = 0

                for other in csp.constraints[scene]:

                    if other not in assignment:
                        connections = connections + 1

                if connections > highest:
                    highest = connections
                    result = scene

        return result

## Step 06: Implement Least Constraining Value (LCV)

In this step you will:

- order values by least restriction
- try least constraining values first

In [6]:
class LCVHeuristic:

    @staticmethod
    def order_values(variable, csp, assignment):

        value_scores = []

        for v in csp.domains[variable]:

            total = 0

            for other in csp.constraints[variable]:

                if other not in assignment:

                    if v in csp.domains[other]:
                        total = total + 1

            value_scores.append((total, v))

        value_scores.sort()

        sorted_values = []

        for s, v in value_scores:
            sorted_values.append(v)

        return sorted_values

## Step 07: Implement Forward Checking Inference

In this step you will:

- remove inconsistent neighboring values
- apply inference after assignments

In [7]:
class Inference:

    @staticmethod
    def forward_checking(csp, variable, value, assignment):

        removed = {}

        for other in csp.constraints[variable]:

            if other not in assignment:

                if value in csp.domains[other]:

                    if other not in removed:
                        removed[other] = []

                    removed[other].append(value)
                    csp.domains[other].remove(value)

                    if len(csp.domains[other]) == 0:

                        for v in removed:
                            for x in removed[v]:
                                csp.domains[v].append(x)

                        return None

        return removed

## Step 08: Implement Backtracking Solver

In this step you will implement:

- recursive backtracking
- assignment generation
- inference integration
- backtracking removal

In [ ]:
class BacktrackingSolver:

    def backtrack(self, assignment, csp):

        if len(assignment) == len(csp.variables):
            return assignment

        var = MRVHeuristic.select_unassigned_variable(csp, assignment)

        actor_list = LCVHeuristic.order_values(var, csp, assignment)

        for actor in actor_list:

            if csp.is_consistent(var, actor, assignment):

                assignment[var] = actor

                removed = Inference.forward_checking(csp, var, actor, assignment)

                if removed is not None:

                    answer = self.backtrack(assignment, csp)

                    if answer is not None:
                        return answer

                    for v in removed:
                        for x in removed[v]:
                            csp.domains[v].append(x)

                del assignment[var]

        return None

## Step 09: Output Writer (Do NOT Modify)

In [9]:
class OutputWriter:

    @staticmethod
    def write_output(solution):

        with open("output.txt", "w") as f:

            if solution is None:

                text = "No valid casting found."

                print(text)

                f.write(text)

            else:

                print("\nFinal Scene Casting:\n")

                f.write("Final Scene Casting:\n\n")

                for scene, actor in solution.items():

                    line = f"{scene} -> {actor}"

                    print(line)

                    f.write(line + "\n")

        print("\nResults written to output.txt")

## Step 11: Main Method (Do NOT Modify)

In [10]:
def main():

    actors, scenes = InputHandler.load_data(
        "actors.txt",
        "scenes.txt"
    )

    csp = FilmCSP()

    for scene in scenes:

        valid_actors = []

        for actor in actors:

            if scenes[scene] in actors[actor]:

                valid_actors.append(actor)

        csp.add_variable(scene, valid_actors)

    solver = BacktrackingSolver()

    solution = solver.backtrack({}, csp)

    OutputWriter.write_output(solution)


if __name__ == "__main__":
    main()


Final Scene Casting:

Scene5 -> Alexander
Scene8 -> Maya
Scene18 -> Noah
Scene11 -> Emma
Scene12 -> Mia
Scene2 -> Olivia
Scene15 -> Liam
Scene7 -> James
Scene20 -> Matthew
Scene10 -> Charlotte
Scene1 -> Amelia
Scene3 -> Ava
Scene16 -> Henry
Scene14 -> Ethan
Scene4 -> Zayn
Scene17 -> Benjamin
Scene6 -> Harper
Scene19 -> Aria
Scene9 -> Leo
Scene13 -> Daniel

Results written to output.txt


## Step 12: Questions

1. Why is CSP useful in film production systems?

2. How does MRV reduce search complexity?

3. Why is Least Constraining Value effective?

4. What is the role of Forward Checking?

5. How can this system scale for large productions?

In [11]:
# Answers

# 1. In film production there are many rules to follow when assigning actors to scenes.
#    CSP handles all these rules at once and finds a correct assignment automatically.
#    Without it we would have to check every possibility by hand which takes a lot of time.

# 2. MRV always picks the scene that has the least number of actors available.
#    If a scene has very few options it is better to assign it first. This way the
#    search fails quickly instead of wasting time going through many other scenes first.

# 3. LCV tries the actor that removes the least choices from other scenes first.
#    By doing this the remaining scenes still have many options available and the
#    solver is less likely to get stuck and have to go back and try again.

# 4. Forward Checking looks ahead after every assignment is made. It removes the
#    assigned actor from all related scenes right away. If any scene ends up with
#    no options left it backtracks immediately without going further down that path.

# 5. For large productions we can connect only scenes that actually share actors
#    instead of connecting every pair of scenes. We can also use stronger methods
#    like AC-3 to reduce the search space before the solver even starts running.